# Module 3.4: Belief Revision

*This notebook applies to **semantic memory** — preferences and facts stored as
Cosmos DB Preference nodes. Episodic events don't get revised (a trip either happened or
it didn't — they expire via TTL instead). Procedural knowledge has its own staleness
mechanism (RAG validation), covered in Notebook 06.*

Facts change. People move cities, switch hotel chains, update dietary preferences.
What happens when an agent's semantic memory says *"Sarah lives in New York"* but
she just told it she moved to Paris?

> **The question**: How should an agent handle semantic facts that change over time?

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, sniffio, certifi
sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider)
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from lifecycle_utils import CosmosBeliefStore
from shared.travel_agent import (
    create_client, SYSTEM_PROMPT, search_flights, search_hotels, get_travel_policy)

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

## Setup: Connect to Cosmos DB

Same semantic memory graph from Module 2.3 / 3.3. Preferences live as nodes with
vector embeddings. The `CosmosBeliefStore` in
[lifecycle_utils.py](lifecycle_utils.py) adds bi-temporal properties
(`valid_from`, `valid_to`, `superseded_by`) directly on those nodes.

In [ ]:
from azure.cosmos.aio import CosmosClient
from shared.semantic_store import SemanticMemoryStore, create_container
from pydantic import SecretStr

embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default")
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token, api_version="2024-02-01")

async def embed(text: str) -> list[float]:
    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

print("Embedding provider ready.")

## The Problem: Facts Change, and Nobody Remembers

Let's walk through a scenario.

**January**: Sarah tells the travel agent "I'm based in NYC." The agent
stores this as a preference. When Sarah asks "Book me a flight to the London
conference," the agent knows to search departures from NYC airports. Works
perfectly.

**February 15**: Sarah tells the agent "I'm relocating to Paris on March 1st."
Now what? The developer has two obvious choices:

---

**Choice 1: Overwrite** — delete "based in NYC", write "based in Paris."

On **February 20** — before the move — Sarah says "Book me a flight to
Chicago next week." The agent searches departures from *Paris*. But Sarah
is still in NYC for another 10 days. She gets CDG→ORD options when she
needed JFK→ORD. The overwrite destroyed the temporal nuance: the old fact
was still true *right now*, and the new fact isn't true *yet*.

Even after March 1st, if Sarah asks "Can you rebook that Chicago trip I
took in February?", the agent has no record she was ever in NYC. It would
search Paris departures for a trip that left from JFK.

**Choice 2: Append** — keep "based in NYC", also add "based in Paris."

On **February 20**, Sarah says "Book me a flight to Chicago." The agent
finds *two* home locations with no way to tell which is current. Does she
fly from NYC or Paris? It might pick one at random, ask a clarifying
question that Sarah already answered, or worse — book from the wrong city
without asking.

After March 1st, the same ambiguity persists forever. The agent has two
facts and no timestamps to resolve which one applies *when*.

---

Neither approach answers the question the agent actually needs to resolve:

> **"Where is Sarah based *right now* — and where *was* she based when
> she took that earlier trip?"**

This is the **temporal audit problem**. The agent needs to know not just
what's true *today*, but what was true *at any point in the past* — and
exactly when one fact replaced another. Without that, it cannot correctly
book future travel *or* reason about past trips.

> **⚠️ A note on memory isolation**: In this scenario, only Sarah's own
> agent uses her memory. No other user or manager can query it. Memory
> isolation (who can read whose data) is a critical design concern covered
> in the **Governance & Security** module. Here, assume all queries are
> same-user — Sarah asking *her* agent about *her* preferences.

**Why vector search can't save you here:**

When you store "Sarah is based in NYC" and later store "Sarah is based in
Paris", both sentences have nearly identical embeddings — same person, same
topic, same structure. Cosine similarity scores them 0.92–0.97 apart.

So the vector index returns **both** with nearly equal scores. It cannot
distinguish "this replaced that" from "these are two similar facts." The
retrieval layer treats a correction as a near-duplicate.

This means RAG serves superseded (outdated) values 15–40% of the time
*(arXiv:2606.26511)* — not because the system is broken, but because
similarity search has no concept of *time* or *supersession*.

## The Solution: SCD Type 2 (Supersession)

Borrowed from data warehousing: every version of a fact is a separate row
with `valid_from` / `valid_to` timestamps. When a new value arrives, the old
row is *retired* (its `valid_to` is set) — never deleted.

```mermaid
stateDiagram-v2
    [*] --> Current : store belief
    Current --> Superseded : new value for same category
    Superseded --> [*] : retained for audit
    Current --> Current : same value confirmed
```

This gives us:
- **Current query** → "Where does Sarah live?" → Paris ✅
- **Time-travel query** → "Where did she live in January?" → NYC ✅
- **Full audit trail** → every change, with timestamps and provenance

In [ ]:
cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
container = await create_container(cosmos)
print("CosmosBeliefStore ready (SCD Type 2)")
print(f"Connected to Cosmos DB: semantic-memory")

beliefs = CosmosBeliefStore(
    SemanticMemoryStore(container, user_id="E001", embed_fn=embed,
                        scope_tag="nb04_belief"),
)
await beliefs.reset()


## The Revision Agent

The agent gets four memory tools — thin wrappers over `CosmosBeliefStore`:

| Tool | Purpose |
|------|---------|
| `remember_belief` | Store a preference, auto-superseding any conflicting current belief |
| `recall_current_beliefs` | Retrieve only currently-valid beliefs (superseded ones filtered out) |
| `recall_beliefs_at_time` | Time-travel: what did we believe on a specific date? |
| `belief_history` | Full SCD Type 2 audit trail for a category |

The agent's instructions tell it to use `source_type='user_assertion'` for
explicit user statements and `'llm_inference'` for anything derived.

In [ ]:
@tool
async def remember_belief(category: str, preference: str,
                          source_type: str = "user_assertion") -> str:
    """Store a belief. Supersedes any conflicting current belief in the same category.
    source_type: 'user_assertion' for explicit statements, 'llm_inference' for derived."""
    r = await beliefs.store(category, preference, source_type)
    sup = f" (superseded: '{r['superseded']}')" if r["superseded"] else ""
    return f"Stored '{preference}' as {r['state']}{sup}."

@tool
async def recall_current_beliefs(query: str) -> str:
    """Recall currently-valid beliefs. Superseded beliefs are excluded."""
    return await beliefs.recall_current(query)

@tool
async def recall_beliefs_at_time(query: str, date: str) -> str:
    """Time-travel: recall beliefs valid on a specific date (YYYY-MM-DD)."""
    return await beliefs.recall_at_time(query, date)

print("Store + recall tools defined.")

In [ ]:
@tool
async def belief_history(category: str) -> str:
    """Full audit trail for a belief category (all versions with timestamps)."""
    rows = await beliefs.history(category)
    if not rows:
        return f"No history for '{category}'."

    lines = []
    print("History tool defined.")

    for r in rows:
        vt = r['valid_to'][:10] if r['valid_to'] else 'present'
        lines.append(f"{r['preference']} ({r['valid_from'][:10]} → {vt})")
    return "\n".join(lines)

In [ ]:
assistant = Agent(
    client=client, name="TravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "You manage the user's long-term preferences with belief revision.\n"
        "- When the user states a preference, IMMEDIATELY call remember_belief "
        "with the appropriate category (e.g. 'home_city', 'hotel_chain', "
        "'home_airport', 'diet').\n"
        "- 'I moved to X' or 'I live in X' → remember_belief(category='home_city', preference=X)\n"
        "- 'X is my airport' → remember_belief(category='home_airport', preference=X)\n"
        "- Use source_type='user_assertion' for explicit statements.\n"
        "- Before recommendations, call recall_current_beliefs.\n"
        "- For historical questions ('where did I live in Jan?'), use "
        "recall_beliefs_at_time.\n"
        "- If the user asks how a preference evolved, use belief_history."),
    tools=[search_flights, search_hotels, get_travel_policy,
           remember_belief, recall_current_beliefs,
           recall_beliefs_at_time, belief_history])
print(f"Agent ready: {assistant.name}")

## Demo: Sarah Moves Cities

We run the exact scenario that breaks overwrite and append-only.
The agent stores preferences, handles a city change (supersession),
and answers both current and historical queries correctly.

In [ ]:
session = AgentSession()

r1 = await assistant.run(
    "I live in New York and my home airport is JFK. "
    "I always stay at Marriott when travelling.",
    session=session)
print("ASSISTANT:", r1.text, "\n")
for p in await beliefs.snapshot():
    print(f"  [{p['state']:11s}] {p['category']:15s} → {p['preference']}")

In [ ]:
r2 = await assistant.run(
    "Big news — I just moved to Paris permanently! "
    "Update my home city. CDG is my new home airport.",
    session=session)
print("ASSISTANT:", r2.text, "\n")
for p in await beliefs.snapshot():
    status = "CURRENT" if p["valid_to"] is None else "SUPERSEDED"
    print(f"  [{status:10s}] {p['category']:15s} → {p['preference']}")

In [ ]:
r3 = await assistant.run(
    "Wait, I need to file an expense from January this year. "
    "Where was I living back then? Which airport would I have used?",
    session=session)
print("ASSISTANT:", r3.text)

## Edge Case: Visiting vs Moving

Not every location mention is a permanent change. The agent's instructions
and the `category` parameter handle this naturally:

- *"I just moved to Paris"* → `remember_belief(category='home_city', ...)` → supersedes NYC
- *"I'm visiting Tokyo next week"* → the agent should NOT call `remember_belief` for `home_city`

The agent reasons about whether a statement represents a permanent change.
Let's test it:

In [ ]:
r4 = await assistant.run(
    "I'm visiting Tokyo next week for a conference. Can you find me a hotel?",
    session=session)
print("ASSISTANT:", r4.text, "\n")

# Verify: home_city should still be Paris, not Tokyo
print("Current beliefs after 'visiting Tokyo':")
for p in await beliefs.snapshot():
    if p["valid_to"] is None:
        print(f"  [{p['category']:15s}] {p['preference']}")

## Audit Trail: Full Belief History

Every change is preserved in Cosmos DB with timestamps. The agent can retrieve
the full evolution of any belief category — useful for compliance, debugging,
and explaining past recommendations.

In [ ]:
r5 = await assistant.run(
    "Show me the full history of where I've lived.",
    session=session)
print("ASSISTANT:", r5.text, "\n")

# Read directly from Cosmos DB for verification
print("Cosmos DB audit trail (home_city):")
for r in await beliefs.history("home_city"):
    vt = r["valid_to"][:10] if r["valid_to"] else "present"
    print(f"  {r['preference']:15s} | {r['valid_from'][:10]} → {vt}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **SCD Type 2** | Every version is a separate node with `valid_from` / `valid_to` |
| **Supersession** | Same category, new value → old node retired, not deleted |
| **Time-travel** | `recall_beliefs_at_time` filters by `valid_from ≤ T < valid_to` |
| **Audit trail** | `belief_history` returns full evolution with provenance |
| **Composability** | Beliefs still have `state` from Module 3.3 (trust promotion) |

1. **Never overwrite** — retire the old value with a `valid_to` timestamp
2. **Never append blindly** — supersession prevents contradictions in recall
3. **Two time dimensions** — valid-time (real world) and transaction-time (when stored)
4. **The agent drives it** — it stores, supersedes, and queries through tools
5. **Same Cosmos DB graph** — no separate store; temporal properties on existing Preference nodes

## Next: Retention & Decay (Notebook 05)

The belief store grows over time. The next notebook implements bounded memory
with scoring-based retention — ensuring old, unused, or redundant beliefs are
gracefully evicted rather than accumulated indefinitely.